# Self-supervised position estimator, n=120 (`tab:app-position`) -- FLAGGED

**This run fed CNT images in [0,1] rather than [-1,1]** (the bug `corrected_pipeline_rerun.ipynb` fixed on 3 Aug); the uncorrected run kept about 24% of source contrast versus 92% after the fix, and `tab:app-position` was never rerun after the fix. Printed output only -- no file in the archive holds these numbers.

**Needs:** DTD, the CNT checkpoint (pre-fix, to reproduce exactly what the paper table shows).

**Maintained runnable path:** none. This notebook (uncorrected) is the only record of `tab:app-position` as published.

In [ ]:
import os
!pip install -q pot

import torch, torch.nn.functional as F, numpy as np, os, sys, glob, random
from pathlib import Path
from PIL import Image
from google.colab import drive
drive.mount('/content/drive')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
BASE = os.environ.get('DATA_ROOT', '/content/drive/MyDrive/DATA_ROOT')  # was hardcoded to the Drive mount; set DATA_ROOT to point at your local copy
DTD_ROOT = f'{BASE}/dtd/images'

os.chdir('/content'); os.system('rm -rf cnt-siga24')
os.system('git clone -q https://github.com/phtu-cs/compositional-neural-textures.git cnt-siga24')
os.system('pip install -q -r cnt-siga24/requirements.in')
os.system('pip install -q "git+https://github.com/facebookresearch/detectron2.git"')
os.system(f'rm -rf /content/cnt-siga24/ckpts; ln -sfn "{BASE}/cnt_ckpts" /content/cnt-siga24/ckpts')
os.chdir('/content/cnt-siga24'); sys.path.insert(0,'/content/cnt-siga24')
from src.inference.core import TexAutoEncoderInference
model = TexAutoEncoderInference(ckpt_path=Path("/content/cnt-siga24/ckpts/finetuned-model256.ckpt"), device=device)
IMG_SIZE = 256
FMS = [round(IMG_SIZE*r) for r in model.generator.feature_map_size_ratios]

def get_gen_inputs(img):
    with torch.no_grad():
        blob = model.encode(img); layout = model.splat(blob)
        return blob, layout["layer_level_layouts"][-1][0]["blob_features"].clone(), \
                     layout["layer_level_layouts"][-1][0]["score_img"].clone()
def decode_interp(blob_ref, bf, si):
    with torch.no_grad():
        layout = model.splat(blob_ref)
        layout["layer_level_layouts"][-1][0]["blob_features"] = bf
        layout["layer_level_layouts"][-1][0]["score_img"] = si
        return model.decode(layout)
def build_style_maps(bf, si, fms):
    sp={}; s=si; sp[s.shape[-1]]=s
    while s.shape[-1]>min(fms):
        s=F.interpolate(s,s.shape[-1]//2,mode="bilinear",align_corners=False); sp[s.shape[-1]]=s
    return {size: torch.einsum("bnc,bnhw->bchw",bf,sp[size]) for size in set(fms)}
def decode_with_style_maps(blob_ref, sm):
    with torch.no_grad():
        layout=model.splat(blob_ref); si=layout["layer_level_layouts"][-1][0]["score_img"]
        fms=[round(si.shape[-1]*r) for r in model.generator.feature_map_size_ratios]
        sp={}; s=si; sp[s.shape[-1]]=s
        while s.shape[-1]>min(fms):
            s=F.interpolate(s,s.shape[-1]//2,mode="bilinear",align_corners=False); sp[s.shape[-1]]=s
        feat_in=sp[fms[0]][:,1:].amax(dim=1)[:,None].repeat(1,model.generator.n_embedding,1,1)
        x=feat_in.clone()
        for i in range(len(fms)):
            x=model.generator.spade_blocks[i](x,sm[fms[i]])
            if i==len(fms)-1: break
            elif fms[i]!=fms[i+1]: x=F.interpolate(x,size=x.shape[-1]*2,mode="bilinear",align_corners=False)
        return torch.tanh(x.unsqueeze(1)).flatten(1,2)
def sinkhorn_transport(src,tgt,eps=0.05,n_iters=100):
    N,M=src.shape[0],tgt.shape[0]; C=torch.cdist(src,tgt).pow(2); K=torch.exp(-C/eps)
    a=torch.ones(N,1,device=src.device)/N; b=torch.ones(M,1,device=src.device)/M; u=torch.ones(N,1,device=src.device)
    for _ in range(n_iters): v=b/(K.T@u+1e-10); u=a/(K@v+1e-10)
    plan=u*K*v.T; plan=plan/(plan.sum(1,keepdim=True)+1e-10); return plan@tgt

def build_d(ia, ib):
    ba,bfa,sia = get_gen_inputs(ia); bb,bfb,sib = get_gen_inputs(ib)
    return {"blob_a":ba,"bf_a":bfa,"si_a":sia,"blob_b":bb,"bf_b":bfb,"si_b":sib}
def interp_ot(d, eta, n_sub=4096):
    sm_a=build_style_maps(d["bf_a"],d["si_a"],FMS); sm_b=build_style_maps(d["bf_b"],d["si_b"],FMS)
    L=max(FMS); src,tgt=sm_a[L][0],sm_b[L][0]; C,H,W=src.shape
    sf=src.permute(1,2,0).reshape(-1,C); tf=tgt.permute(1,2,0).reshape(-1,C); N=sf.shape[0]
    i_s=torch.randperm(N,device=device)[:n_sub]; i_t=torch.randperm(N,device=device)[:n_sub]
    with torch.no_grad(): tt=sinkhorn_transport(sf[i_s],tf[i_t])
    nn=torch.cdist(sf,sf[i_s]).argmin(1)
    interp=((1-eta)*sf+eta*tt[nn]).reshape(H,W,C).permute(2,0,1).unsqueeze(0)
    sm={s:(interp if s==L else F.interpolate(interp,size=s,mode="bilinear",align_corners=False)) for s in set(FMS)}
    return decode_with_style_maps(d["blob_a"], sm)
def interp_lin_pixel(d, eta):
    sm_a=build_style_maps(d["bf_a"],d["si_a"],FMS); sm_b=build_style_maps(d["bf_b"],d["si_b"],FMS)
    return decode_with_style_maps(d["blob_a"], {s:(1-eta)*sm_a[s]+eta*sm_b[s] for s in set(FMS)})
def interp_lin_texton(d, eta):
    return decode_interp(d["blob_a"], (1-eta)*d["bf_a"]+eta*d["bf_b"], (1-eta)*d["si_a"]+eta*d["si_b"])
def dec_to_np(dec):
    return ((dec.squeeze(0).detach().cpu().clamp(-1,1)+1)/2).permute(1,2,0).numpy()
def np_to_tensor(a):
    return torch.from_numpy(a).float().permute(2,0,1).unsqueeze(0).to(device)

# --- two frozen observers ---
import timm
from torchvision.models import inception_v3, Inception_V3_Weights
dino = timm.create_model('vit_small_patch14_dinov2.lvd142m', pretrained=True, num_classes=0).to(device).eval()
_inc = inception_v3(weights=Inception_V3_Weights.IMAGENET1K_V1, aux_logits=True).to(device).eval()
_inc.fc = torch.nn.Identity()
for p in list(dino.parameters())+list(_inc.parameters()): p.requires_grad_(False)
_M=torch.tensor([0.485,0.456,0.406],device=device).view(1,3,1,1)
_S=torch.tensor([0.229,0.224,0.225],device=device).view(1,3,1,1)

@torch.no_grad()
def _feats(net, size, arrs, bs=16):
    out=[]
    for i in range(0,len(arrs),bs):
        b=torch.stack([torch.from_numpy(a).float().permute(2,0,1) for a in arrs[i:i+bs]]).to(device)
        b=F.interpolate(b,size=size,mode='bilinear',align_corners=False); b=(b-_M)/_S
        out.append(F.normalize(net(b),dim=1).cpu().numpy())
    return np.concatenate(out)
dino_feats = lambda a: _feats(dino, 518, a)
inc_feats  = lambda a: _feats(_inc, 299, a)

print("ready on", device)

In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

N_PAIRS = 120
ETAS    = [0.0, 0.25, 0.5, 0.75, 1.0]
METHODS = ['ot','pixel_linear','texton_linear']
N_AUG   = 12          # augmented views per endpoint to fit the boundary

random.seed(0)
cats = sorted([c for c in os.listdir(DTD_ROOT) if os.path.isdir(f'{DTD_ROOT}/{c}')])
def dtd_path(c,s): return random.Random(s).choice(sorted(glob.glob(f'{DTD_ROOT}/{c}/*.jpg')))

rng = np.random.default_rng(0)
pairs=[]
while len(pairs) < N_PAIRS:
    a,b = random.sample(cats,2)
    pairs.append((dtd_path(a,len(pairs)), dtd_path(b,len(pairs)+7000)))

def load_256(path):
    pil = Image.open(path).convert('RGB')
    W,H = pil.size
    x,y = (W-IMG_SIZE)//2, (H-IMG_SIZE)//2
    return np.asarray(pil.crop((x,y,x+IMG_SIZE,y+IMG_SIZE))).astype(np.float32)/255.

def augment(arr, n, seed):
    """Crops+flips of the SAME texture: gives the boundary within-class variation."""
    r = np.random.default_rng(seed)
    pil = Image.fromarray((arr*255).astype(np.uint8))
    out=[arr]
    for _ in range(n-1):
        s = r.integers(180, 250)
        x,y = r.integers(0, IMG_SIZE-s+1, size=2)
        c = pil.crop((x,y,x+s,y+s)).resize((IMG_SIZE,IMG_SIZE), Image.BILINEAR)
        c = np.asarray(c).astype(np.float32)/255.
        if r.random() < 0.5: c = c[:, ::-1].copy()
        if r.random() < 0.5: c = c[::-1, :].copy()
        out.append(c)
    return out

In [ ]:
def fit_boundary(featfn, arrsA, arrsB):
    X = np.vstack([featfn(arrsA), featfn(arrsB)])
    y = np.array([0]*len(arrsA) + [1]*len(arrsB))
    sc = StandardScaler().fit(X)
    clf = LogisticRegression(max_iter=2000, C=1.0).fit(sc.transform(X), y)
    return lambda arrs: clf.predict_proba(sc.transform(featfn(arrs)))[:,1], clf, sc, X, y

for k in range(2):
    pa, pb = pairs[k]
    imA, imB = load_256(pa), load_256(pb)
    augA, augB = augment(imA, N_AUG, 100+k), augment(imB, N_AUG, 200+k)
    for tag, featfn in [('dino', dino_feats), ('inc', inc_feats)]:
        p_of, clf, sc, X, y = fit_boundary(featfn, augA, augB)
        # held-out-ish check: probabilities on the training views
        pA, pB = p_of(augA), p_of(augB)
        # round-trip control: does a reconstruction of A still read as A?
        rt = dec_to_np(interp_ot(build_d(np_to_tensor(imA), np_to_tensor(imA)), 0.0))
        p_rt = p_of([rt])[0]
        print(f"pair {k} [{tag}]  p(B) on A-views {pA.mean():.3f}±{pA.std():.3f}   "
              f"on B-views {pB.mean():.3f}±{pB.std():.3f}   round-trip(A) {p_rt:.3f}")
    print()

In [ ]:
for k in range(3):
    pa, pb = pairs[k]
    imA, imB = load_256(pa), load_256(pb)
    augA, augB = augment(imA, N_AUG, 100+k), augment(imB, N_AUG, 200+k)
    d = build_d(np_to_tensor(imA), np_to_tensor(imB))
    mids = {m: [dec_to_np({'ot':interp_ot,'pixel_linear':interp_lin_pixel,
                           'texton_linear':interp_lin_texton}[m](d,e)) for e in ETAS]
            for m in METHODS}
    for tag, featfn in [('dino', dino_feats), ('inc', inc_feats)]:
        for C in [1.0, 0.01]:
            X = np.vstack([featfn(augA), featfn(augB)])
            y = np.array([0]*len(augA)+[1]*len(augB))
            sc = StandardScaler().fit(X)
            clf = LogisticRegression(max_iter=2000, C=C).fit(sc.transform(X), y)
            ps = clf.predict_proba(sc.transform(featfn(mids['ot'])))[:,1]
            print(f"pair {k} [{tag}] C={C:<5} OT p(B) across eta: " +
                  " ".join(f"{p:.3f}" for p in ps))
    print()

In [ ]:
def recon(arr):
    t = np_to_tensor(arr)
    return dec_to_np(interp_ot(build_d(t, t), 0.0))

for k in range(3):
    pa, pb = pairs[k]
    imA, imB = load_256(pa), load_256(pb)
    rA = [recon(a) for a in augment(imA, 6, 100+k)]
    rB = [recon(a) for a in augment(imB, 6, 200+k)]
    d = build_d(np_to_tensor(imA), np_to_tensor(imB))
    mids = [dec_to_np(interp_ot(d, e)) for e in ETAS]
    for tag, featfn in [('dino', dino_feats), ('inc', inc_feats)]:
        X = np.vstack([featfn(rA), featfn(rB)])
        y = np.array([0]*len(rA)+[1]*len(rB))
        sc = StandardScaler().fit(X)
        clf = LogisticRegression(max_iter=2000, C=0.1).fit(sc.transform(X), y)
        ps = clf.predict_proba(sc.transform(featfn(mids)))[:,1]
        print(f"pair {k} [{tag}] OT p(B): " + " ".join(f"{p:.3f}" for p in ps))
    print()

In [ ]:
for k in range(3):
    pa, pb = pairs[k]
    imA, imB = load_256(pa), load_256(pb)
    rA = [recon(a) for a in augment(imA, 6, 100+k)]
    rB = [recon(a) for a in augment(imB, 6, 200+k)]
    d = build_d(np_to_tensor(imA), np_to_tensor(imB))
    fns = {'ot':interp_ot,'pixel_linear':interp_lin_pixel,'texton_linear':interp_lin_texton}
    for tag, featfn in [('dino', dino_feats), ('inc', inc_feats)]:
        X = np.vstack([featfn(rA), featfn(rB)])
        y = np.array([0]*len(rA)+[1]*len(rB))
        sc = StandardScaler().fit(X); clf = LogisticRegression(max_iter=2000, C=0.1).fit(sc.transform(X), y)
        print(f"pair {k} [{tag}]")
        for m in METHODS:
            mids = [dec_to_np(fns[m](d, e)) for e in ETAS]
            ps = clf.predict_proba(sc.transform(featfn(mids)))[:,1]
            print(f"    {m:14} " + " ".join(f"{p:.3f}" for p in ps))
    print()

In [ ]:
import pandas as pd
from scipy.stats import wilcoxon

N_PAIRS_RUN = 120
N_ANCHOR    = 6          # reconstructed views per endpoint used to fit the boundary
C_REG       = 0.1
fns = {'ot':interp_ot, 'pixel_linear':interp_lin_pixel, 'texton_linear':interp_lin_texton}

def recon(arr):
    t = np_to_tensor(arr)
    return dec_to_np(interp_ot(build_d(t, t), 0.0))

records = []
for k in range(N_PAIRS_RUN):
    pa, pb = pairs[k]
    imA, imB = load_256(pa), load_256(pb)
    rA = [recon(a) for a in augment(imA, N_ANCHOR, 100+k)]
    rB = [recon(a) for a in augment(imB, N_ANCHOR, 200+k)]
    d  = build_d(np_to_tensor(imA), np_to_tensor(imB))
    mids = {m: [dec_to_np(fns[m](d, e)) for e in ETAS] for m in METHODS}

    rec = dict(pair=k)
    for tag, featfn in [('dino', dino_feats), ('inc', inc_feats)]:
        X = np.vstack([featfn(rA), featfn(rB)])
        y = np.array([0]*len(rA) + [1]*len(rB))
        sc = StandardScaler().fit(X)
        clf = LogisticRegression(max_iter=2000, C=C_REG).fit(sc.transform(X), y)

        for m in METHODS:
            ps = clf.predict_proba(sc.transform(featfn(mids[m])))[:,1]
            p0, p1 = ps[0], ps[-1]
            rec[f'{tag}_{m}_p0']  = p0
            rec[f'{tag}_{m}_p1']  = p1
            rec[f'{tag}_{m}_span'] = p1 - p0                      # endpoint recovery
            rec[f'{tag}_{m}_praw'] = ps[ETAS.index(0.5)]          # raw midpoint reading
            if p1 - p0 > 0.2:                                     # usable span
                pn = (ps - p0) / (p1 - p0)
                rec[f'{tag}_{m}_pmid'] = pn[ETAS.index(0.5)]      # normalised placement
                rec[f'{tag}_{m}_bias'] = abs(pn[ETAS.index(0.5)] - 0.5)
                rec[f'{tag}_{m}_mono'] = float(np.mean(np.diff(pn) > 0))
            else:
                rec[f'{tag}_{m}_pmid'] = np.nan
                rec[f'{tag}_{m}_bias'] = np.nan
                rec[f'{tag}_{m}_mono'] = np.nan
    records.append(rec)
    if k % 15 == 0: print(f"  {k}/{N_PAIRS_RUN}")

dfp = pd.DataFrame(records)
dfp.to_csv(f'{BASE}/position_estimator_n{N_PAIRS_RUN}.csv', index=False)

for tag, name in [('dino','DINOv2'), ('inc','InceptionV3')]:
    print(f"\n=== {name} (n={N_PAIRS_RUN}) ===")
    print(f"  {'method':16} {'span (p1-p0)':>13} {'usable':>8} {'raw p(.5)':>11} "
          f"{'norm p(.5)':>11} {'|bias|':>8} {'mono':>7}")
    for m in METHODS:
        usable = dfp[f'{tag}_{m}_pmid'].notna().mean()
        print(f"  {m:16} {dfp[f'{tag}_{m}_span'].median():13.3f} {100*usable:7.0f}% "
              f"{dfp[f'{tag}_{m}_praw'].median():11.3f} {dfp[f'{tag}_{m}_pmid'].median():11.3f} "
              f"{dfp[f'{tag}_{m}_bias'].median():8.3f} {dfp[f'{tag}_{m}_mono'].mean():7.2f}")

    print(f"  --- OT vs baselines ---")
    for base in ['pixel_linear','texton_linear']:
        for key, lab, alt in [('span','endpoint recovery','greater'),
                              ('bias','midpoint bias','less')]:
            a, b = dfp[f'{tag}_ot_{key}'], dfp[f'{tag}_{base}_{key}']
            ok = a.notna() & b.notna()
            w = int(((a[ok]<b[ok]) if alt=='less' else (a[ok]>b[ok])).sum())
            p = wilcoxon(a[ok], b[ok], alternative=alt).pvalue
            print(f"    vs {base:14} [{lab:18}]: {w}/{ok.sum()}  p={p:.2e}  "
                  f"{'OT WINS' if w>ok.sum()/2 else 'OT loses'}")

print(f"\n=== observer agreement ===")
for m in METHODS:
    ok = dfp[f'dino_{m}_bias'].notna() & dfp[f'inc_{m}_bias'].notna()
    r = np.corrcoef(dfp.loc[ok,f'dino_{m}_bias'], dfp.loc[ok,f'inc_{m}_bias'])[0,1]
    print(f"  {m:16} per-pair correlation of |bias|: {r:.3f}")

In [ ]:
N_SYM = 60
rows = []

for k in range(N_SYM):
    pa, pb = pairs[k]
    imA, imB = load_256(pa), load_256(pb)
    rA = [recon(a) for a in augment(imA, N_ANCHOR, 100+k)]
    rB = [recon(a) for a in augment(imB, N_ANCHOR, 200+k)]

    d_ab = build_d(np_to_tensor(imA), np_to_tensor(imB))
    d_ba = build_d(np_to_tensor(imB), np_to_tensor(imA))

    mids_ab = [dec_to_np(interp_ot(d_ab, e)) for e in ETAS]     # A -> B
    mids_ba = [dec_to_np(interp_ot(d_ba, e)) for e in ETAS]     # B -> A

    rec = dict(pair=k)
    for tag, featfn in [('dino', dino_feats), ('inc', inc_feats)]:
        X = np.vstack([featfn(rA), featfn(rB)])
        y = np.array([0]*len(rA) + [1]*len(rB))
        sc = StandardScaler().fit(X)
        clf = LogisticRegression(max_iter=2000, C=C_REG).fit(sc.transform(X), y)
        pf = lambda arrs: clf.predict_proba(sc.transform(featfn(arrs)))[:,1]

        ps_ab = pf(mids_ab)                       # p(B) along A->B
        ps_ba = pf(mids_ba)                       # p(B) along B->A: starts high, ends low

        # normalise each to its own endpoints, in the direction it travels
        def norm(ps, lo, hi):
            return (ps - lo)/(hi - lo) if abs(hi-lo) > 0.2 else np.full_like(ps, np.nan)

        n_ab = norm(ps_ab, ps_ab[0],  ps_ab[-1])          # 0 at A, 1 at B
        n_ba = norm(ps_ba, ps_ba[-1], ps_ba[0])           # also 0 at A, 1 at B

        mid_ab = n_ab[ETAS.index(0.5)]
        mid_ba = n_ba[ETAS.index(0.5)]

        rec[f'{tag}_mid_ab']   = mid_ab                    # placement, A->B
        rec[f'{tag}_mid_ba']   = mid_ba                    # placement, B->A (same A-B axis)
        rec[f'{tag}_span_ab']  = ps_ab[-1] - ps_ab[0]
        rec[f'{tag}_span_ba']  = ps_ba[0]  - ps_ba[-1]
        # if the bias is directional, the two midpoints sit on OPPOSITE sides of 0.5
        rec[f'{tag}_sym_gap']  = mid_ab - (1.0 - mid_ba)   # 0 = order-invariant
    rows.append(rec)
    if k % 10 == 0: print(f"  {k}/{N_SYM}")

dsym = pd.DataFrame(rows)
dsym.to_csv(f'{BASE}/ot_direction_symmetry_n{N_SYM}.csv', index=False)

for tag, name in [('dino','DINOv2'), ('inc','InceptionV3')]:
    ab, ba = dsym[f'{tag}_mid_ab'], dsym[f'{tag}_mid_ba']
    ok = ab.notna() & ba.notna()
    print(f"\n=== {name} (n={ok.sum()}) ===")
    print(f"  midpoint of A->B, on the A-B axis : {ab[ok].median():.3f}")
    print(f"  midpoint of B->A, on the A-B axis : {ba[ok].median():.3f}")
    print(f"  span A->B {dsym[f'{tag}_span_ab'][ok].median():.3f} | "
          f"span B->A {dsym[f'{tag}_span_ba'][ok].median():.3f}")
    print(f"  symmetry gap (0 = order-invariant) : {dsym[f'{tag}_sym_gap'][ok].median():+.3f}")
    frac = float((ab[ok] < 0.5).mean()), float((ba[ok] < 0.5).mean())
    print(f"  fraction of midpoints on A's side : A->B {frac[0]:.2f}   B->A {frac[1]:.2f}")

In [ ]:
imA, imB = load_256(pairs[0][0]), load_256(pairs[0][1])
bA, bfA, siA = get_gen_inputs(np_to_tensor(imA))
bB, bfB, siB = get_gen_inputs(np_to_tensor(imB))
print("blob type:", type(bA))
if isinstance(bA, dict):
    for k,v in bA.items():
        print(f"  {k}: {type(v).__name__} {tuple(v.shape) if hasattr(v,'shape') else v}")
else:
    print("  shape:", getattr(bA, 'shape', None))
print("\nblob_features:", tuple(bfA.shape), " score_img:", tuple(siA.shape))

In [ ]:
k = 0
imA, imB = load_256(pairs[k][0]), load_256(pairs[k][1])
tA, tB = np_to_tensor(imA), np_to_tensor(imB)
d = build_d(tA, tB)

sm_a = build_style_maps(d["bf_a"], d["si_a"], FMS)
sm_b = build_style_maps(d["bf_b"], d["si_b"], FMS)
L = max(FMS)
src, tgt = sm_a[L][0], sm_b[L][0]
C,H,W = src.shape
sf = src.permute(1,2,0).reshape(-1,C); tf = tgt.permute(1,2,0).reshape(-1,C)
i_s = torch.randperm(sf.shape[0], device=device)[:4096]
i_t = torch.randperm(tf.shape[0], device=device)[:4096]
tt = sinkhorn_transport(sf[i_s], tf[i_t])
nn = torch.cdist(sf, sf[i_s]).argmin(1)
interp = (0.5*sf + 0.5*tt[nn]).reshape(H,W,C).permute(2,0,1).unsqueeze(0)
sm = {s:(interp if s==L else F.interpolate(interp,size=s,mode="bilinear",align_corners=False)) for s in set(FMS)}

out_a = dec_to_np(decode_with_style_maps(d["blob_a"], sm))
out_b = dec_to_np(decode_with_style_maps(d["blob_b"], sm))

rA = [recon(a) for a in augment(imA, N_ANCHOR, 100+k)]
rB = [recon(a) for a in augment(imB, N_ANCHOR, 200+k)]
for tag, featfn in [('dino', dino_feats), ('inc', inc_feats)]:
    X = np.vstack([featfn(rA), featfn(rB)]); y = np.array([0]*len(rA)+[1]*len(rB))
    sc = StandardScaler().fit(X); clf = LogisticRegression(max_iter=2000, C=C_REG).fit(sc.transform(X), y)
    p = clf.predict_proba(sc.transform(featfn([out_a, out_b])))[:,1]
    print(f"{tag}: same style maps, decoded with A's blob -> p(B)={p[0]:.3f} | with B's blob -> p(B)={p[1]:.3f}")

In [ ]:
k = 0
imA, imB = load_256(pairs[k][0]), load_256(pairs[k][1])
d = build_d(np_to_tensor(imA), np_to_tensor(imB))
rA = [recon(a) for a in augment(imA, N_ANCHOR, 100+k)]
rB = [recon(a) for a in augment(imB, N_ANCHOR, 200+k)]

for tag, featfn in [('dino', dino_feats), ('inc', inc_feats)]:
    X = np.vstack([featfn(rA), featfn(rB)]); y = np.array([0]*len(rA)+[1]*len(rB))
    sc = StandardScaler().fit(X)
    clf = LogisticRegression(max_iter=2000, C=C_REG).fit(sc.transform(X), y)
    for ns in [2048, 4096, 8192, 16384]:
        mids = [dec_to_np(interp_ot(d, e, n_sub=ns)) for e in ETAS]
        ps = clf.predict_proba(sc.transform(featfn(mids)))[:,1]
        print(f"{tag} n_sub={ns:6d}: " + " ".join(f"{p:.3f}" for p in ps))
    print()

In [ ]:
def interp_ot_sym(d, eta, n_sub=4096):
    """Symmetric OT: transport A->B and B->A, blend each, average the style maps.
    Both endpoints pass through the same nearest-neighbour approximation, so neither
    is represented exactly while the other is approximated."""
    sm_a = build_style_maps(d["bf_a"], d["si_a"], FMS)
    sm_b = build_style_maps(d["bf_b"], d["si_b"], FMS)
    L = max(FMS)
    A, B = sm_a[L][0], sm_b[L][0]
    C,H,W = A.shape
    af = A.permute(1,2,0).reshape(-1,C)
    bf = B.permute(1,2,0).reshape(-1,C)
    N = af.shape[0]

    ia = torch.randperm(N, device=device)[:n_sub]
    ib = torch.randperm(N, device=device)[:n_sub]

    with torch.no_grad():
        t_ab = sinkhorn_transport(af[ia], bf[ib])     # A's samples pushed to B
        t_ba = sinkhorn_transport(bf[ib], af[ia])     # B's samples pushed to A

    nn_a = torch.cdist(af, af[ia]).argmin(1)          # every A pixel -> its sample
    nn_b = torch.cdist(bf, bf[ib]).argmin(1)          # every B pixel -> its sample

    # forward path: A exact -> B approximated
    fwd = (1-eta)*af + eta*t_ab[nn_a]
    # backward path: B exact -> A approximated, expressed on the same eta axis
    bwd = eta*bf + (1-eta)*t_ba[nn_b]

    interp = 0.5*(fwd + bwd)
    interp = interp.reshape(H,W,C).permute(2,0,1).unsqueeze(0)
    sm = {s:(interp if s==L else F.interpolate(interp,size=s,mode="bilinear",align_corners=False))
          for s in set(FMS)}
    return decode_with_style_maps(d["blob_a"], sm)

# --- quick look on 3 pairs before committing ---
for k in range(3):
    imA, imB = load_256(pairs[k][0]), load_256(pairs[k][1])
    d = build_d(np_to_tensor(imA), np_to_tensor(imB))
    rA = [recon(a) for a in augment(imA, N_ANCHOR, 100+k)]
    rB = [recon(a) for a in augment(imB, N_ANCHOR, 200+k)]
    for tag, featfn in [('dino', dino_feats), ('inc', inc_feats)]:
        X = np.vstack([featfn(rA), featfn(rB)]); y = np.array([0]*len(rA)+[1]*len(rB))
        sc = StandardScaler().fit(X)
        clf = LogisticRegression(max_iter=2000, C=C_REG).fit(sc.transform(X), y)
        for name, fn in [('ot', interp_ot), ('ot_sym', interp_ot_sym)]:
            ps = clf.predict_proba(sc.transform(featfn([dec_to_np(fn(d,e)) for e in ETAS])))[:,1]
            print(f"pair {k} [{tag}] {name:7} " + " ".join(f"{p:.3f}" for p in ps))
    print()

In [ ]:
N_RUN = 120
fns_v = {'ot':interp_ot, 'ot_sym':interp_ot_sym,
         'pixel_linear':interp_lin_pixel, 'texton_linear':interp_lin_texton}
rows = []

for k in range(N_RUN):
    imA, imB = load_256(pairs[k][0]), load_256(pairs[k][1])
    d = build_d(np_to_tensor(imA), np_to_tensor(imB))
    rA = [recon(a) for a in augment(imA, N_ANCHOR, 100+k)]
    rB = [recon(a) for a in augment(imB, N_ANCHOR, 200+k)]
    mids = {m: [dec_to_np(fn(d, e)) for e in ETAS] for m, fn in fns_v.items()}

    rec = dict(pair=k)
    for tag, featfn in [('dino', dino_feats), ('inc', inc_feats)]:
        X = np.vstack([featfn(rA), featfn(rB)]); y = np.array([0]*len(rA)+[1]*len(rB))
        sc = StandardScaler().fit(X)
        clf = LogisticRegression(max_iter=2000, C=C_REG).fit(sc.transform(X), y)
        for m in fns_v:
            ps = clf.predict_proba(sc.transform(featfn(mids[m])))[:,1]
            p0, p1 = ps[0], ps[-1]
            rec[f'{tag}_{m}_p0']   = p0
            rec[f'{tag}_{m}_span'] = p1 - p0
            if p1 - p0 > 0.2:
                pn = (ps - p0)/(p1 - p0)
                rec[f'{tag}_{m}_pmid'] = pn[ETAS.index(0.5)]
                rec[f'{tag}_{m}_bias'] = abs(pn[ETAS.index(0.5)] - 0.5)
                rec[f'{tag}_{m}_mono'] = float(np.mean(np.diff(pn) > 0))
            else:
                for key in ['pmid','bias','mono']: rec[f'{tag}_{m}_{key}'] = np.nan
    rows.append(rec)
    if k % 15 == 0: print(f"  {k}/{N_RUN}")

dfv = pd.DataFrame(rows)
dfv.to_csv(f'{BASE}/ot_symmetric_position_n{N_RUN}.csv', index=False)

for tag, name in [('dino','DINOv2'), ('inc','InceptionV3')]:
    print(f"\n=== {name} (n={N_RUN}) ===")
    print(f"  {'method':16} {'p0':>7} {'span':>8} {'norm p(.5)':>11} {'|bias|':>8} {'mono':>7}")
    for m in fns_v:
        print(f"  {m:16} {dfv[f'{tag}_{m}_p0'].median():7.3f} {dfv[f'{tag}_{m}_span'].median():8.3f} "
              f"{dfv[f'{tag}_{m}_pmid'].median():11.3f} {dfv[f'{tag}_{m}_bias'].median():8.3f} "
              f"{dfv[f'{tag}_{m}_mono'].mean():7.2f}")
    print("  --- ot_sym vs others (midpoint bias, lower better) ---")
    for base in ['ot','pixel_linear','texton_linear']:
        a, b = dfv[f'{tag}_ot_sym_bias'], dfv[f'{tag}_{base}_bias']
        ok = a.notna() & b.notna()
        w = int((a[ok] < b[ok]).sum())
        p = wilcoxon(a[ok], b[ok], alternative='less').pvalue
        print(f"    vs {base:14}: {w}/{ok.sum()}  p={p:.2e}  "
              f"{'SYM WINS' if w>ok.sum()/2 else 'SYM loses'}")

In [ ]:
def interp_ot_sym(d, eta, n_sub=4096):
    """Symmetric OT: transport A->B and B->A, blend each on the same eta axis,
    average the style maps. Neither endpoint is privileged."""
    sm_a = build_style_maps(d["bf_a"], d["si_a"], FMS)
    sm_b = build_style_maps(d["bf_b"], d["si_b"], FMS)
    L = max(FMS)
    A, B = sm_a[L][0], sm_b[L][0]
    C,H,W = A.shape
    af = A.permute(1,2,0).reshape(-1,C)
    bf = B.permute(1,2,0).reshape(-1,C)
    N = af.shape[0]
    ia = torch.randperm(N, device=device)[:n_sub]
    ib = torch.randperm(N, device=device)[:n_sub]
    with torch.no_grad():
        t_ab = sinkhorn_transport(af[ia], bf[ib])
        t_ba = sinkhorn_transport(bf[ib], af[ia])
    nn_a = torch.cdist(af, af[ia]).argmin(1)
    nn_b = torch.cdist(bf, bf[ib]).argmin(1)
    fwd = (1-eta)*af + eta*t_ab[nn_a]
    bwd = eta*bf + (1-eta)*t_ba[nn_b]
    interp = (0.5*(fwd + bwd)).reshape(H,W,C).permute(2,0,1).unsqueeze(0)
    sm = {s:(interp if s==L else F.interpolate(interp,size=s,mode="bilinear",align_corners=False))
          for s in set(FMS)}
    return decode_with_style_maps(d["blob_a"], sm)

In [ ]:
TARGET_MEAN, TARGET_RMS = 0.5, 0.18

def gen_oriented_noise(f0, theta_deg, size=256, f_bw=0.35, th_bw_deg=18.0, seed=0):
    """Stationary texture: white noise bandpass-filtered at radial frequency f0
    (cycles/image) and orientation theta. Parameters known by construction."""
    rng = np.random.default_rng(seed)
    n = rng.standard_normal((size, size))
    Fn = np.fft.fftshift(np.fft.fft2(n))
    cy = cx = size // 2
    yy, xx = np.mgrid[0:size, 0:size]
    fy, fx = (yy - cy).astype(float), (xx - cx).astype(float)
    r = np.sqrt(fx**2 + fy**2) + 1e-9
    ang = np.arctan2(fy, fx)
    radial = np.exp(-((np.log(r / f0))**2) / (2 * f_bw**2))
    th0 = np.deg2rad(theta_deg)
    dth = np.angle(np.exp(1j * 2 * (ang - th0))) / 2.0
    angular = np.exp(-(dth**2) / (2 * np.deg2rad(th_bw_deg)**2))
    filt = radial * angular
    filt[cy, cx] = 0.0
    img = np.real(np.fft.ifft2(np.fft.ifftshift(Fn * filt)))
    img = (img - img.mean()) / (img.std() + 1e-9)
    img = np.clip(img * TARGET_RMS + TARGET_MEAN, 0, 1)
    return np.repeat(img[:, :, None], 3, axis=2).astype(np.float32)

def estimate_params(rgb):
    """Recover (radial frequency, orientation) from the power spectrum."""
    g = rgb.mean(axis=2).astype(float)
    g = g - g.mean()
    P = np.abs(np.fft.fftshift(np.fft.fft2(g)))**2
    H, W = P.shape; cy, cx = H//2, W//2
    yy, xx = np.mgrid[0:H, 0:W]
    fy, fx = (yy-cy).astype(float), (xx-cx).astype(float)
    r = np.sqrt(fx**2 + fy**2)
    m = (r > 2) & (r < min(H, W)/2 - 1)
    w = P[m]
    if w.sum() <= 0: return np.nan, np.nan
    f_est = float((r[m]*w).sum() / w.sum())
    a = np.arctan2(fy[m], fx[m])
    c = (w*np.cos(2*a)).sum(); s = (w*np.sin(2*a)).sum()
    th_est = float(np.degrees(0.5*np.arctan2(s, c)) % 180.0)
    return f_est, th_est

def ang_err(a, b):
    d = abs(a - b) % 180.0
    return min(d, 180.0 - d)

# sanity
for f0, th in [(12, 0), (28, 60)]:
    im = gen_oriented_noise(f0, th, seed=int(f0))
    fe, te = estimate_params(im)
    print(f"  true f={f0} θ={th}  ->  est f={fe:.1f} θ={te:.0f}")

In [ ]:
import pandas as pd
from scipy.stats import wilcoxon

N_PAIRS    = 200
F_LO, F_HI = 8.0, 40.0
TH_JITTER  = 6.0
METHODS    = ['ot', 'ot_sym', 'pixel_linear', 'texton_linear', 'crossfade']
FNS = {'ot':interp_ot, 'ot_sym':interp_ot_sym,
       'pixel_linear':interp_lin_pixel, 'texton_linear':interp_lin_texton}

def rms_contrast(rgb):
    return float(rgb.mean(axis=2).astype(float).std())

def spectral_bimodality(rgb, fa_e, fb_e, w=0.18):
    g = rgb.mean(axis=2).astype(float); g = g - g.mean()
    P = np.abs(np.fft.fftshift(np.fft.fft2(g)))**2
    H, W = P.shape; cy, cx = H//2, W//2
    yy, xx = np.mgrid[0:H, 0:W]
    r = np.sqrt((xx-cx)**2 + (yy-cy)**2)
    def band(f):
        m = (r > f*(1-w)) & (r < f*(1+w)); return float(P[m].sum())
    return (band(fa_e) + band(fb_e)) / (band(np.sqrt(fa_e*fb_e)) + 1e-9)

rng = np.random.default_rng(0)
records = []
for i in range(N_PAIRS):
    fa, fb = np.exp(rng.uniform(np.log(F_LO), np.log(F_HI), size=2))
    if fb < fa: fa, fb = fb, fa
    if fb/fa < 1.6:
        fb = fa*1.6
        if fb > F_HI: fa, fb = F_HI/1.6, F_HI
    f_mid_true = np.sqrt(fa*fb)
    th = rng.uniform(0,180)
    th_a = th + rng.uniform(-TH_JITTER, TH_JITTER)
    th_b = th + rng.uniform(-TH_JITTER, TH_JITTER)
    sa, sb, sm_ = (int(rng.integers(1e6)) for _ in range(3))
    imA  = gen_oriented_noise(fa, th_a, seed=sa)
    imB  = gen_oriented_noise(fb, th_b, seed=sb)
    imGT = gen_oriented_noise(f_mid_true, th, seed=sm_)

    f_gt, th_gt = estimate_params(imGT); c_gt = rms_contrast(imGT)
    fa_e, _ = estimate_params(imA); fb_e, _ = estimate_params(imB)

    d = build_d(np_to_tensor(imA), np_to_tensor(imB))
    mids = {m: dec_to_np(FNS[m](d, 0.5)) for m in FNS}
    mids['crossfade'] = 0.5*imA + 0.5*imB

    rec = dict(pair=i, fa=fa, fb=fb, fa_est=fa_e, fb_est=fb_e, f_gt_est=f_gt,
               bimod_gt=spectral_bimodality(imGT, fa_e, fb_e))
    for m, img in mids.items():
        fe, te = estimate_params(img)
        rec[f'{m}_f']        = fe
        rec[f'{m}_f_relerr'] = abs(fe - f_gt)/f_gt
        rec[f'{m}_th_err']   = ang_err(te, th_gt)
        rec[f'{m}_c_relerr'] = abs(rms_contrast(img) - c_gt)/c_gt
        rec[f'{m}_bimod']    = spectral_bimodality(img, fa_e, fb_e)
    records.append(rec)
    if i % 20 == 0: print(f"  {i}/{N_PAIRS}")

df = pd.DataFrame(records)
df.to_csv(f'{BASE}/parametric_freq_symcompare_n{N_PAIRS}.csv', index=False)

fa_est, fb_est = df['fa_est'].values, df['fb_est'].values
t_imp = {}
for m in METHODS + ['gt']:
    col = df['f_gt_est'] if m=='gt' else df[f'{m}_f']
    t_imp[m] = (np.log(col.values) - np.log(fa_est))/(np.log(fb_est) - np.log(fa_est))

print(f"\n=== FREQUENCY (n={N_PAIRS}) ===")
print(f"  {'method':16} {'|Δf|/f ↓':>10} {'|t-0.5| ↓':>11} {'bimod':>8} {'Δθ°':>7}")
print(f"  {'ground truth':16} {'-':>10} {np.median(abs(t_imp['gt']-0.5)):11.3f} "
      f"{df['bimod_gt'].median():8.2f} {'-':>7}")
for m in METHODS:
    print(f"  {m:16} {df[f'{m}_f_relerr'].median():10.3f} "
          f"{np.median(abs(t_imp[m]-0.5)):11.3f} {df[f'{m}_bimod'].median():8.2f} "
          f"{df[f'{m}_th_err'].median():7.1f}")

print(f"\n=== ot_sym vs each (paired) ===")
for base in ['ot','pixel_linear','texton_linear','crossfade']:
    for key, lab in [('f_relerr','recovery'), ('bimod','superposition')]:
        a, b = df[f'ot_sym_{key}'], df[f'{base}_{key}']
        w = int((a<b).sum()); p = wilcoxon(a, b, alternative='less').pvalue
        print(f"  vs {base:14} [{lab:14}]: {w}/{len(a)}  p={p:.2e}  "
              f"{'SYM WINS' if w>len(a)/2 else 'SYM loses'}")
    a = np.abs(t_imp['ot_sym']-0.5); b = np.abs(t_imp[base]-0.5)
    w = int((a<b).sum()); p = wilcoxon(a, b, alternative='less').pvalue
    print(f"  vs {base:14} [{'placement':14}]: {w}/{len(a)}  p={p:.2e}  "
          f"{'SYM WINS' if w>len(a)/2 else 'SYM loses'}")

In [ ]:
N_PAIRS  = 200
F_FIXED  = 20.0
DTH_MIN, DTH_MAX = 25.0, 75.0
METHODS  = ['ot', 'ot_sym', 'pixel_linear', 'texton_linear', 'crossfade']

def circ_mid(a, b):
    d = ((b - a + 90.0) % 180.0) - 90.0
    return (a + d/2.0) % 180.0
def signed_ang(a, b):
    return ((a - b + 90.0) % 180.0) - 90.0

def orientation_bimodality(rgb, th_a_e, th_b_e, w_deg=12.0):
    g = rgb.mean(axis=2).astype(float); g = g - g.mean()
    P = np.abs(np.fft.fftshift(np.fft.fft2(g)))**2
    H, W = P.shape; cy, cx = H//2, W//2
    yy, xx = np.mgrid[0:H, 0:W]
    fy, fx = (yy-cy).astype(float), (xx-cx).astype(float)
    r = np.sqrt(fx**2+fy**2); ang = np.degrees(np.arctan2(fy, fx)) % 180.0
    ring = (r > F_FIXED*0.6) & (r < F_FIXED*1.5)
    def band(th):
        d = np.abs(((ang - th + 90) % 180) - 90)
        return float(P[ring & (d < w_deg)].sum())
    return (band(th_a_e) + band(th_b_e)) / (band(circ_mid(th_a_e, th_b_e)) + 1e-9)

rng = np.random.default_rng(1)
records = []
for i in range(N_PAIRS):
    th_a = rng.uniform(0, 180)
    dth  = rng.uniform(DTH_MIN, DTH_MAX) * rng.choice([-1, 1])
    th_b = (th_a + dth) % 180.0
    th_gt = circ_mid(th_a, th_b)

    sa, sb, sm_ = (int(rng.integers(1e6)) for _ in range(3))
    imA  = gen_oriented_noise(F_FIXED, th_a,  seed=sa)
    imB  = gen_oriented_noise(F_FIXED, th_b,  seed=sb)
    imGT = gen_oriented_noise(F_FIXED, th_gt, seed=sm_)

    f_gt_e, th_gt_e = estimate_params(imGT)
    _, th_a_e = estimate_params(imA)
    _, th_b_e = estimate_params(imB)
    c_gt = rms_contrast(imGT)

    d = build_d(np_to_tensor(imA), np_to_tensor(imB))
    mids = {m: dec_to_np(FNS[m](d, 0.5)) for m in FNS}
    mids['crossfade'] = 0.5*imA + 0.5*imB

    rec = dict(pair=i, dth=abs(dth), th_a_est=th_a_e, th_b_est=th_b_e,
               th_gt_est=th_gt_e, f_gt_est=f_gt_e,
               bimod_gt=orientation_bimodality(imGT, th_a_e, th_b_e))
    span = signed_ang(th_b_e, th_a_e)
    for m, img in mids.items():
        fe, te = estimate_params(img)
        rec[f'{m}_th_err']  = ang_err(te, th_gt_e)
        rec[f'{m}_f_drift'] = abs(fe - f_gt_e)/f_gt_e          # should stay ~0
        rec[f'{m}_c_relerr']= abs(rms_contrast(img) - c_gt)/c_gt
        rec[f'{m}_bimod']   = orientation_bimodality(img, th_a_e, th_b_e)
        rec[f'{m}_t_imp']   = (signed_ang(te, th_a_e)/span) if abs(span) > 1e-6 else np.nan
    records.append(rec)
    if i % 20 == 0: print(f"  {i}/{N_PAIRS}")

dfo = pd.DataFrame(records)
dfo.to_csv(f'{BASE}/parametric_orient_symcompare_n{N_PAIRS}.csv', index=False)

print(f"\n=== ORIENTATION (n={N_PAIRS}), frequency fixed at {F_FIXED} ===")
print(f"  {'method':16} {'Δθ° ↓':>8} {'|t-0.5| ↓':>11} {'bimod':>8} {'f drift ↓':>10}")
print(f"  {'ground truth':16} {'-':>8} {'-':>11} {dfo['bimod_gt'].median():8.2f} {'-':>10}")
for m in METHODS:
    dev = np.abs(dfo[f'{m}_t_imp'] - 0.5)
    print(f"  {m:16} {dfo[f'{m}_th_err'].median():8.2f} {np.nanmedian(dev):11.3f} "
          f"{dfo[f'{m}_bimod'].median():8.2f} {dfo[f'{m}_f_drift'].median():10.3f}")

print(f"\n=== ot_sym vs each (paired) ===")
for base in ['ot','pixel_linear','texton_linear','crossfade']:
    for key, lab in [('th_err','orientation'), ('bimod','superposition')]:
        a, b = dfo[f'ot_sym_{key}'], dfo[f'{base}_{key}']
        ok = a.notna() & b.notna()
        w = int((a[ok]<b[ok]).sum()); p = wilcoxon(a[ok], b[ok], alternative='less').pvalue
        print(f"  vs {base:14} [{lab:14}]: {w}/{ok.sum()}  p={p:.2e}  "
              f"{'SYM WINS' if w>ok.sum()/2 else 'SYM loses'}")
    a = np.abs(dfo['ot_sym_t_imp']-0.5); b = np.abs(dfo[f'{base}_t_imp']-0.5)
    ok = a.notna() & b.notna()
    w = int((a[ok]<b[ok]).sum()); p = wilcoxon(a[ok], b[ok], alternative='less').pvalue
    print(f"  vs {base:14} [{'placement':14}]: {w}/{ok.sum()}  p={p:.2e}  "
          f"{'SYM WINS' if w>ok.sum()/2 else 'SYM loses'}")

print(f"\n=== by endpoint gap ===")
print(f"  {'Δθ range':12} {'n':>4} " + " ".join(f"{m[:9]:>10}" for m in METHODS))
for lo, hi in [(25,40),(40,60),(60,75)]:
    s = dfo[(dfo.dth >= lo) & (dfo.dth < hi)]
    if len(s) < 5: continue
    print(f"  {lo}-{hi}°       {len(s):4d} " + " ".join(f"{s[f'{m}_th_err'].median():10.2f}" for m in METHODS))

In [ ]:
import os
!pip install -q pot

import torch, torch.nn.functional as F, numpy as np, os, sys, glob, random
from pathlib import Path
from PIL import Image
from google.colab import drive
drive.mount('/content/drive')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
BASE = os.environ.get('DATA_ROOT', '/content/drive/MyDrive/DATA_ROOT')  # was hardcoded to the Drive mount; set DATA_ROOT to point at your local copy
DTD_ROOT = f'{BASE}/dtd/images'

os.chdir('/content'); os.system('rm -rf cnt-siga24')
os.system('git clone -q https://github.com/phtu-cs/compositional-neural-textures.git cnt-siga24')
os.system('pip install -q -r cnt-siga24/requirements.in')
os.system('pip install -q "git+https://github.com/facebookresearch/detectron2.git"')
os.system(f'rm -rf /content/cnt-siga24/ckpts; ln -sfn "{BASE}/cnt_ckpts" /content/cnt-siga24/ckpts')
os.chdir('/content/cnt-siga24'); sys.path.insert(0,'/content/cnt-siga24')
from src.inference.core import TexAutoEncoderInference
model = TexAutoEncoderInference(ckpt_path=Path("/content/cnt-siga24/ckpts/finetuned-model256.ckpt"), device=device)
IMG_SIZE = 256
FMS = [round(IMG_SIZE*r) for r in model.generator.feature_map_size_ratios]

def get_gen_inputs(img):
    with torch.no_grad():
        blob = model.encode(img); layout = model.splat(blob)
        return blob, layout["layer_level_layouts"][-1][0]["blob_features"].clone(), \
                     layout["layer_level_layouts"][-1][0]["score_img"].clone()
def decode_interp(blob_ref, bf, si):
    with torch.no_grad():
        layout = model.splat(blob_ref)
        layout["layer_level_layouts"][-1][0]["blob_features"] = bf
        layout["layer_level_layouts"][-1][0]["score_img"] = si
        return model.decode(layout)
def build_style_maps(bf, si, fms):
    sp={}; s=si; sp[s.shape[-1]]=s
    while s.shape[-1]>min(fms):
        s=F.interpolate(s,s.shape[-1]//2,mode="bilinear",align_corners=False); sp[s.shape[-1]]=s
    return {size: torch.einsum("bnc,bnhw->bchw",bf,sp[size]) for size in set(fms)}
def decode_with_style_maps(blob_ref, sm):
    with torch.no_grad():
        layout=model.splat(blob_ref); si=layout["layer_level_layouts"][-1][0]["score_img"]
        fms=[round(si.shape[-1]*r) for r in model.generator.feature_map_size_ratios]
        sp={}; s=si; sp[s.shape[-1]]=s
        while s.shape[-1]>min(fms):
            s=F.interpolate(s,s.shape[-1]//2,mode="bilinear",align_corners=False); sp[s.shape[-1]]=s
        feat_in=sp[fms[0]][:,1:].amax(dim=1)[:,None].repeat(1,model.generator.n_embedding,1,1)
        x=feat_in.clone()
        for i in range(len(fms)):
            x=model.generator.spade_blocks[i](x,sm[fms[i]])
            if i==len(fms)-1: break
            elif fms[i]!=fms[i+1]: x=F.interpolate(x,size=x.shape[-1]*2,mode="bilinear",align_corners=False)
        return torch.tanh(x.unsqueeze(1)).flatten(1,2)
def sinkhorn_transport(src,tgt,eps=0.05,n_iters=100):
    N,M=src.shape[0],tgt.shape[0]; C=torch.cdist(src,tgt).pow(2); K=torch.exp(-C/eps)
    a=torch.ones(N,1,device=src.device)/N; b=torch.ones(M,1,device=src.device)/M; u=torch.ones(N,1,device=src.device)
    for _ in range(n_iters): v=b/(K.T@u+1e-10); u=a/(K@v+1e-10)
    plan=u*K*v.T; plan=plan/(plan.sum(1,keepdim=True)+1e-10); return plan@tgt

def build_d(ia, ib):
    ba,bfa,sia = get_gen_inputs(ia); bb,bfb,sib = get_gen_inputs(ib)
    return {"blob_a":ba,"bf_a":bfa,"si_a":sia,"blob_b":bb,"bf_b":bfb,"si_b":sib}

def interp_ot(d, eta, n_sub=4096):
    sm_a=build_style_maps(d["bf_a"],d["si_a"],FMS); sm_b=build_style_maps(d["bf_b"],d["si_b"],FMS)
    L=max(FMS); src,tgt=sm_a[L][0],sm_b[L][0]; C,H,W=src.shape
    sf=src.permute(1,2,0).reshape(-1,C); tf=tgt.permute(1,2,0).reshape(-1,C); N=sf.shape[0]
    i_s=torch.randperm(N,device=device)[:n_sub]; i_t=torch.randperm(N,device=device)[:n_sub]
    with torch.no_grad(): tt=sinkhorn_transport(sf[i_s],tf[i_t])
    nn=torch.cdist(sf,sf[i_s]).argmin(1)
    interp=((1-eta)*sf+eta*tt[nn]).reshape(H,W,C).permute(2,0,1).unsqueeze(0)
    sm={s:(interp if s==L else F.interpolate(interp,size=s,mode="bilinear",align_corners=False)) for s in set(FMS)}
    return decode_with_style_maps(d["blob_a"], sm)

def interp_ot_sym(d, eta, n_sub=4096):
    sm_a=build_style_maps(d["bf_a"],d["si_a"],FMS); sm_b=build_style_maps(d["bf_b"],d["si_b"],FMS)
    L=max(FMS); A,B=sm_a[L][0],sm_b[L][0]; C,H,W=A.shape
    af=A.permute(1,2,0).reshape(-1,C); bf=B.permute(1,2,0).reshape(-1,C); N=af.shape[0]
    ia=torch.randperm(N,device=device)[:n_sub]; ib=torch.randperm(N,device=device)[:n_sub]
    with torch.no_grad():
        t_ab=sinkhorn_transport(af[ia],bf[ib]); t_ba=sinkhorn_transport(bf[ib],af[ia])
    nn_a=torch.cdist(af,af[ia]).argmin(1); nn_b=torch.cdist(bf,bf[ib]).argmin(1)
    fwd=(1-eta)*af+eta*t_ab[nn_a]; bwd=eta*bf+(1-eta)*t_ba[nn_b]
    interp=(0.5*(fwd+bwd)).reshape(H,W,C).permute(2,0,1).unsqueeze(0)
    sm={s:(interp if s==L else F.interpolate(interp,size=s,mode="bilinear",align_corners=False)) for s in set(FMS)}
    return decode_with_style_maps(d["blob_a"], sm)

def interp_lin_pixel(d, eta):
    sm_a=build_style_maps(d["bf_a"],d["si_a"],FMS); sm_b=build_style_maps(d["bf_b"],d["si_b"],FMS)
    return decode_with_style_maps(d["blob_a"], {s:(1-eta)*sm_a[s]+eta*sm_b[s] for s in set(FMS)})
def interp_lin_texton(d, eta):
    return decode_interp(d["blob_a"], (1-eta)*d["bf_a"]+eta*d["bf_b"], (1-eta)*d["si_a"]+eta*d["si_b"])

def to_pil(dec):
    a=((dec.squeeze(0).detach().cpu().clamp(-1,1)+1)/2).permute(1,2,0).numpy()
    return Image.fromarray((a*255).astype(np.uint8))

# --- realism score observer (InceptionV3) ---
from torchvision.models import inception_v3, Inception_V3_Weights
_inc = inception_v3(weights=Inception_V3_Weights.IMAGENET1K_V1, aux_logits=True).to(device).eval()
_inc.fc = torch.nn.Identity()
for p in _inc.parameters(): p.requires_grad_(False)
_M=torch.tensor([0.485,0.456,0.406],device=device).view(1,3,1,1)
_S=torch.tensor([0.229,0.224,0.225],device=device).view(1,3,1,1)
@torch.no_grad()
def inc_feats(pils, bs=32):
    out=[]
    for i in range(0,len(pils),bs):
        xs=[torch.from_numpy(np.asarray(p.convert('RGB').resize((256,256))).copy()).float().permute(2,0,1)/255.
            for p in pils[i:i+bs]]
        b=torch.stack(xs).to(device)
        b=F.interpolate(b,size=299,mode='bilinear',align_corners=False); b=(b-_M)/_S
        out.append(_inc(b).cpu().numpy())
    return np.concatenate(out)

print("ready on", device)

In [ ]:
from scipy.stats import wilcoxon
random.seed(1)
cats = sorted([c for c in os.listdir(DTD_ROOT) if os.path.isdir(f'{DTD_ROOT}/{c}')])
def dtd_path(c,s): return random.Random(s).choice(sorted(glob.glob(f'{DTD_ROOT}/{c}/*.jpg')))

N = 200
pairs=[]
while len(pairs)<N:
    a,b=random.sample(cats,2)
    pairs.append((dtd_path(a,len(pairs)), dtd_path(b,len(pairs)+7000)))

def pil_to_tensor(pil):
    a=np.asarray(pil.resize((IMG_SIZE,IMG_SIZE))).copy()
    return torch.from_numpy(a).float().div(255).permute(2,0,1).unsqueeze(0).to(device)

# real-texture manifold
imgs=[Image.open(dtd_path(random.choice(cats),50000+i)).convert('RGB') for i in range(400)]
Phi=inc_feats(imgs); Phi=Phi[~np.isnan(Phi).any(axis=1)]
D=np.linalg.norm(Phi[:,None]-Phi[None,:],axis=2); np.fill_diagonal(D,np.inf)
rad=np.sort(D,axis=1)[:,2]; keep=rad<=np.median(rad)
Phi_r, radii = Phi[keep], rad[keep]
def realism(phi):
    d=np.linalg.norm(Phi_r-phi,axis=1)+1e-9; return float(np.max(radii/d))
print(f"manifold: {len(Phi_r)} points")

FNS={'ot':interp_ot,'ot_sym':interp_ot_sym,
     'pixel_linear':interp_lin_pixel,'texton_linear':interp_lin_texton}
mids={m:[] for m in FNS}
import time; t0=time.time()
for k,(pa,pb) in enumerate(pairs):
    ia=pil_to_tensor(Image.open(pa).convert('RGB')); ib=pil_to_tensor(Image.open(pb).convert('RGB'))
    d=build_d(ia,ib)
    for m,fn in FNS.items(): mids[m].append(to_pil(fn(d,0.5)))
    if k%40==0: print(f"  {k}/{N}  ({(time.time()-t0)/60:.1f}m)")

sc={m: np.array([realism(f) for f in inc_feats(mids[m])]) for m in FNS}
np.save(f'{BASE}/color_control_otsym_n{N}.npy', sc, allow_pickle=True)

print(f"\n=== REALISM, colour condition (n={N}) ===")
for m in ['pixel_linear','texton_linear','ot','ot_sym']:
    print(f"  {m:14} mean R {sc[m].mean():.3f}   %R>=1 {100*(sc[m]>=1).mean():.0f}%")

print(f"\n=== ot_sym vs each (paired) ===")
for base in ['ot','pixel_linear','texton_linear']:
    a,b = sc['ot_sym'], sc[base]
    w=int((a>b).sum()); p=wilcoxon(a,b,alternative='greater').pvalue
    print(f"  vs {base:14}: {w}/{N}  p={p:.2e}  {'SYM WINS' if w>N/2 else 'SYM loses'}")

print(f"\n=== reference: previous run had OT vs pixel 114/200 (p=0.004), OT vs texton significant ===")
a,b = sc['ot'], sc['pixel_linear']
print(f"  ot vs pixel_linear this run: {int((a>b).sum())}/{N}  "
      f"p={wilcoxon(a,b,alternative='greater').pvalue:.2e}")